# 📡 Telco Customer Churn Analysis
### A Data Science Mini Project: Why do customers leave, and who is most at risk?

**Author:** Ayush Verma | BCA, Chandigarh University | Data Science & Analytics

---
## 1. Problem Statement
Customer churn (customers leaving the service) directly hurts a telecom company's revenue, and keeping an existing customer is cheaper than acquiring a new one.
This project analyses a real-world telecom dataset to:
1. Understand **how many** customers churn and **who** they are
2. Find the **key factors** behind churn through EDA and visualisation
3. Build a simple **baseline model** to predict churn
4. Give **business recommendations** to reduce churn

## 2. Dataset
**Telco Customer Churn** (IBM sample dataset, public): 7,043 customers, 21 columns covering demographics, subscribed services, contract, billing and the target `Churn`.

## 3. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.titleweight": "bold"})
PALETTE = {"No": "#2a9d8f", "Yes": "#e76f51"}
RANDOM_STATE = 42

## 4. Load the Data

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
try:
    df = pd.read_csv("data/Telco-Customer-Churn.csv")   # local copy (GitHub repo)
except FileNotFoundError:
    df = pd.read_csv(URL)                                 # online copy (Colab)
print("Shape:", df.shape)
df.head()

## 5. Data Cleaning & Overview

In [ ]:
df.info()

In [ ]:
# TotalCharges is stored as text because new customers have a blank value
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
print("Missing values after conversion:\n", df.isna().sum()[df.isna().sum() > 0])

# These rows are customers with tenure = 0 (brand new) -> total charges is 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
print("Duplicate rows:", df.duplicated().sum())
df.describe().round(2)

## 6. Exploratory Data Analysis

### 6.1 How many customers churn?

In [ ]:
churn_rate = (df["Churn"] == "Yes").mean()
counts = df["Churn"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(data=df, x="Churn", hue="Churn", palette=PALETTE, legend=False, ax=axes[0])
axes[0].set_title("Customers by Churn Status")
for p in axes[0].patches:
    axes[0].annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom")
axes[1].pie(counts.values, labels=["Stayed", "Churned"], autopct="%1.1f%%", startangle=90,
            colors=[PALETTE["No"], PALETTE["Yes"]], wedgeprops=dict(edgecolor="white", linewidth=2))
axes[1].set_title("Churn Share")
plt.tight_layout(); plt.show()
print(f"💡 Insight: {churn_rate:.1%} of customers churned. The data is imbalanced (about 3 stayers for every churner).")

### 6.2 Contract type, the strongest churn signal

In [ ]:
def churn_by(col, order=None, title=None, rotate=0):
    rate = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean() * 100)
    if order: rate = rate.reindex(order)
    fig, ax = plt.subplots()
    sns.barplot(x=rate.index, y=rate.values, hue=rate.index, palette="viridis", legend=False, ax=ax)
    ax.axhline(churn_rate * 100, color="red", linestyle="--", label=f"Average ({churn_rate:.1%})")
    ax.set_title(title or f"Churn Rate by {col}")
    ax.set_ylabel("Churn Rate (%)"); ax.set_xlabel("")
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width()/2, p.get_height()),
                    ha="center", va="bottom")
    plt.xticks(rotation=rotate); ax.legend(); plt.tight_layout(); plt.show()
    return rate

r = churn_by("Contract", ["Month-to-month", "One year", "Two year"], "Churn Rate by Contract Type")
print(f"💡 Insight: Month-to-month customers churn at {r['Month-to-month']:.1f}% vs only {r['Two year']:.1f}% on two-year contracts.")

### 6.3 Internet service & payment method

In [ ]:
r1 = churn_by("InternetService", title="Churn Rate by Internet Service")
r2 = churn_by("PaymentMethod", title="Churn Rate by Payment Method", rotate=15)
print(f"💡 Insight: Fibre-optic users churn at {r1['Fiber optic']:.1f}% (vs {r1['DSL']:.1f}% for DSL), and electronic-check payers churn at {r2['Electronic check']:.1f}%, the highest of all payment methods.")

### 6.4 Tenure: how long do customers stay?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, palette=PALETTE, multiple="stack", ax=axes[0])
axes[0].set_title("Tenure Distribution by Churn"); axes[0].set_xlabel("Tenure (months)")
sns.boxplot(data=df, x="Churn", y="tenure", hue="Churn", palette=PALETTE, legend=False, ax=axes[1])
axes[1].set_title("Tenure: Churned vs Stayed")
plt.tight_layout(); plt.show()
med = df.groupby("Churn")["tenure"].median()
print(f"💡 Insight: Median tenure of churned customers is {med['Yes']:.0f} months vs {med['No']:.0f} months for loyal ones. Most churn happens early in the customer lifecycle.")

### 6.5 Monthly charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.kdeplot(data=df, x="MonthlyCharges", hue="Churn", palette=PALETTE, fill=True, alpha=0.4, ax=axes[0])
axes[0].set_title("Monthly Charges Distribution")
sns.scatterplot(data=df.sample(1500, random_state=RANDOM_STATE), x="tenure", y="MonthlyCharges",
                hue="Churn", palette=PALETTE, alpha=0.6, ax=axes[1])
axes[1].set_title("Tenure vs Monthly Charges")
plt.tight_layout(); plt.show()
m = df.groupby("Churn")["MonthlyCharges"].median()
print(f"💡 Insight: Churned customers pay more per month (median ${m['Yes']:.1f} vs {m['No']:.1f}), so pricing/value perception plays a role.")

### 6.6 Demographics & add-on services

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, ["gender", "SeniorCitizen", "Partner"]):
    rate = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean() * 100)
    sns.barplot(x=rate.index, y=rate.values, hue=rate.index, palette="viridis", legend=False, ax=ax)
    ax.set_title(f"Churn % by {col}"); ax.set_ylabel("Churn Rate (%)"); ax.set_xlabel("")
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width()/2, p.get_height()), ha="center", va="bottom")
plt.tight_layout(); plt.show()

services = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
sv = df[df["InternetService"] != "No"]
svc = pd.DataFrame({s: sv.groupby(s)["Churn"].apply(lambda x: (x == "Yes").mean() * 100) for s in services}).T
svc[["Yes", "No"]].plot(kind="bar", figsize=(10, 5), color=["#2a9d8f", "#e76f51"])
plt.title("Churn Rate: With vs Without Each Add-on Service"); plt.ylabel("Churn Rate (%)")
plt.legend(["Has service", "No service"]); plt.xticks(rotation=20); plt.tight_layout(); plt.show()
print("💡 Insight: Gender has almost no effect. Senior citizens churn far more, and customers WITH Online Security / Tech Support churn much less.")

### 6.7 Correlation of numeric features

In [ ]:
num = df[["tenure", "MonthlyCharges", "TotalCharges"]].copy()
num["Churn_flag"] = (df["Churn"] == "Yes").astype(int)
sns.heatmap(num.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap"); plt.show()

## 7. Baseline Predictive Model
Goal: predict whether a customer will churn, so the business can act *before* they leave.

In [ ]:
data = df.drop(columns=["customerID"]).copy()
data["Churn"] = (data["Churn"] == "Yes").astype(int)
data = pd.get_dummies(data, drop_first=True)

X = data.drop("Churn", axis=1)
y = data["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE),
}
rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    rep = classification_report(y_test, pred, output_dict=True)
    rows.append({"Model": name, "Accuracy": round(accuracy_score(y_test, pred), 3),
                 "Churn Recall": round(rep["1"]["recall"], 3),
                 "Churn Precision": round(rep["1"]["precision"], 3),
                 "ROC-AUC": round(roc_auc_score(y_test, proba), 3)})
results = pd.DataFrame(rows)
results

In [ ]:
best = models["Logistic Regression"]
cm = confusion_matrix(y_test, best.predict(X_test))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Stayed", "Churned"], yticklabels=["Stayed", "Churned"])
plt.title("Confusion Matrix: Logistic Regression"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

coef = pd.Series(best.named_steps["logisticregression"].coef_[0], index=X.columns)
top = pd.concat([coef.nlargest(7), coef.nsmallest(7)]).sort_values()
top.plot(kind="barh", figsize=(9, 6), color=["#2a9d8f" if v < 0 else "#e76f51" for v in top])
plt.title("Top Factors: Increase (red) / Reduce (green) Churn Risk"); plt.xlabel("Coefficient (scaled)")
plt.tight_layout(); plt.show()

**Note:** `class_weight="balanced"` is used because only ~27% of customers churn. Accuracy alone would be misleading, so **recall on the churn class** and **ROC-AUC** are the metrics that matter here.

## 8. Key Findings

| # | Finding |
|---|---------|
| 1 | About **26.5%** of customers churn (imbalanced data) |
| 2 | **Month-to-month contracts** churn ~43% vs ~3% on two-year contracts |
| 3 | **Fibre-optic** users and **electronic-check** payers churn the most |
| 4 | **New customers** (low tenure) are the most likely to leave |
| 5 | Customers paying **higher monthly charges** churn more |
| 6 | **Senior citizens** and customers **without partners** churn more; gender has no effect |
| 7 | Add-ons like **Online Security** and **Tech Support** are linked to lower churn |

## 9. Business Recommendations
1. **Push long-term contracts** with discounts or perks for month-to-month users.
2. **Run an onboarding programme** for the first 6–12 months, when churn risk is highest.
3. **Bundle Tech Support & Online Security** for fibre customers.
4. **Encourage auto-pay** (bank transfer / credit card) instead of electronic cheques.
5. **Use the churn model** to flag high-risk customers for retention offers.

## 10. Limitations & Next Steps
- The dataset is a single snapshot, so we can show association, **not causation**.
- Next: hyperparameter tuning, cross-validation, SMOTE for imbalance, and an interactive dashboard.

**Tools:** Python • Pandas • NumPy • Matplotlib • Seaborn • Scikit-learn

*Author: Ayush Verma*